# 🧬 Génération & Optimisation de Molécules par IA
### NVIDIA MolMIM × SMILES × Machine Learning

**Objectif :** Explorer l'espace chimique, générer de nouvelles molécules candidates et prédire leurs propriétés physico-chimiques (logP, QED, SAS) grâce au ML.

---
**⚡ Optimisations anti-crash :**
- Sampling stratifié pour visualisations (max 5K points)
- Traitement par chunks pour les calculs lourds
- Suppression des variables inutiles + `gc.collect()`
- `matplotlib` backend non-interactif
- RDKit en mode silencieux

## 📦 0. Installation des dépendances

In [1]:
# Installation (à exécuter une seule fois)
import subprocess, sys

packages = [
    'rdkit',
    'scikit-learn',
    'xgboost',
    'lightgbm',
    'shap',
    'matplotlib',
    'seaborn',
    'pandas',
    'numpy',
    'tqdm',
    'joblib'
]

for pkg in packages:
    try:
        __import__(pkg.replace('-', '_'))
        print(f'✅ {pkg} déjà installé')
    except ImportError:
        print(f'📥 Installation de {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'✅ {pkg} installé')

✅ rdkit déjà installé
📥 Installation de scikit-learn...
✅ scikit-learn installé
✅ xgboost déjà installé
✅ lightgbm déjà installé
✅ shap déjà installé
✅ matplotlib déjà installé
✅ seaborn déjà installé
✅ pandas déjà installé
✅ numpy déjà installé
✅ tqdm déjà installé
✅ joblib déjà installé


## ⚙️ 1. Configuration globale & imports

In [17]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # ← Backend non-interactif : évite les crashes mémoire
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
import joblib
import json
import time

# RDKit (mode silencieux)
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, AllChem, DataStructs, Draw
from rdkit.Chem import rdMolDescriptors
RDLogger.DisableLog('rdApp.*')  # ← Supprime les warnings RDKit

# ML
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import xgboost as xgb
import lightgbm as lgb
import shap

# ─── CONFIGURATION GLOBALE ────────────────────────────────────────────────────
CONFIG = {
    # Données
    'data_path': 'Zinc250k.csv',     # ← Changez ce chemin
    'smiles_col': 'smiles',
    'target_cols': ['logP', 'qed', 'SAS'],
    
    # Anti-crash : limites de sampling
    'viz_sample_size': 5_000,            # Max points pour visualisations
    'fingerprint_chunk_size': 10_000,    # Taille des chunks pour calcul fingerprints
    'tsne_sample_size': 3_000,           # t-SNE est O(n²) → très limité !
    'pca_sample_size': 10_000,
    
    # ML
    'test_size': 0.2,
    'random_state': 42,
    'cv_folds': 5,
    
    # Fingerprints
    'fp_radius': 2,
    'fp_nbits': 1024,
    
    # Outputs
    'output_dir': 'outputs/',
    'models_dir': 'models/',
    'figures_dir': 'figures/',
}

# Créer les dossiers
for d in [CONFIG['output_dir'], CONFIG['models_dir'], CONFIG['figures_dir']]:
    Path(d).mkdir(parents=True, exist_ok=True)

# Style global des plots
plt.style.use('seaborn-v0_8-darkgrid')
COLORS = {'logP': '#3B82F6', 'qed': '#10B981', 'SAS': '#F59E0B', 'accent': '#8B5CF6'}

print('✅ Configuration chargée')
print(f'   Python {sys.version.split()[0]} | Pandas {pd.__version__} | NumPy {np.__version__}')
print(f'   Sample viz: {CONFIG["viz_sample_size"]:,} | Chunk: {CONFIG["fingerprint_chunk_size"]:,}')

✅ Configuration chargée
   Python 3.10.20 | Pandas 2.3.3 | NumPy 2.2.6
   Sample viz: 5,000 | Chunk: 10,000


## 📂 2. Chargement & Nettoyage du Dataset

In [18]:
def load_dataset(path: str, smiles_col: str = 'smiles') -> pd.DataFrame:
    """
    Chargement optimisé avec dtypes explicites pour réduire la mémoire RAM.
    """
    print(f'📂 Chargement de {path}...')
    
    # Lecture avec dtypes optimisés (économise ~30% de RAM)
    df = pd.read_csv(
        path,
        dtype={
            smiles_col: 'string',
            'logP': 'float32',
            'qed': 'float32',
            'SAS': 'float32',
        },
        low_memory=True
    )
    
    print(f'   Shape initiale : {df.shape}')
    print(f'   Mémoire RAM    : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
    return df


def clean_dataset(df: pd.DataFrame, smiles_col: str = 'smiles') -> pd.DataFrame:
    """
    Nettoyage : supprime SMILES invalides, doublons, valeurs aberrantes.
    Traitement en chunks pour éviter le crash mémoire.
    """
    print('🧹 Nettoyage du dataset...')
    n_initial = len(df)
    
    # 1. Suppression des valeurs manquantes
    df = df.dropna(subset=[smiles_col] + CONFIG['target_cols'])
    print(f'   Après dropna        : {len(df):,} ({n_initial - len(df):,} supprimés)')
    
    # 2. Validation SMILES par chunks (évite le crash sur 250K lignes)
    print('   Validation SMILES (par chunks)...')
    valid_mask = []
    chunk_size = CONFIG['fingerprint_chunk_size']
    
    for i in tqdm(range(0, len(df), chunk_size), desc='   Validation'):
        chunk = df[smiles_col].iloc[i:i+chunk_size]
        valid_mask.extend([Chem.MolFromSmiles(s) is not None for s in chunk])
    
    df = df[valid_mask].copy()
    print(f'   Après validation    : {len(df):,}')
    
    # 3. Suppression des doublons
    df = df.drop_duplicates(subset=[smiles_col])
    print(f'   Après déduplication : {len(df):,}')
    
    # 4. Filtrage des outliers (IQR x3 pour être conservateur)
    for col in CONFIG['target_cols']:
        Q1, Q3 = df[col].quantile(0.01), df[col].quantile(0.99)
        df = df[(df[col] >= Q1) & (df[col] <= Q3)]
    print(f'   Après outliers      : {len(df):,}')
    
    # Reset index
    df = df.reset_index(drop=True)
    print(f'✅ Dataset propre : {df.shape} | RAM : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
    return df


df_raw = load_dataset(CONFIG['data_path'])
df = clean_dataset(df_raw, CONFIG['smiles_col'])
del df_raw; gc.collect()

print(f'\n📊 Aperçu :')
df.head(3)

📂 Chargement de Zinc250k.csv...
   Shape initiale : (249455, 4)
   Mémoire RAM    : 28.5 MB
🧹 Nettoyage du dataset...
   Après dropna        : 249,455 (0 supprimés)
   Validation SMILES (par chunks)...


   Validation: 100%|██████████| 25/25 [01:02<00:00,  2.51s/it]


   Après validation    : 249,455
   Après déduplication : 249,455
   Après outliers      : 234,783
✅ Dataset propre : (234783, 4) | RAM : 26.8 MB

📊 Aperçu :


,smiles,logP,qed,SAS
0,CC(C)(C)c1ccc2occ(CC(=O)Nc3ccccc3F)c2c1\n,5.05060,0.702012,2.084095
1,C[C@@H]1CC(Nc2cncc(-c3nncn3C)c2)C[C@@H](C)C1\n,3.11370,0.928975,3.432004
2,N#Cc1ccc(-c2ccc(O[C@@H](C(=O)N3CCCC3)c3ccccc3)...,4.96778,0.599682,2.470633


## 📊 3. Analyse Exploratoire (EDA) — Optimisée Anti-Crash

In [19]:
def sample_for_viz(df: pd.DataFrame, n: int = None, stratify_col: str = None) -> pd.DataFrame:
    """
    Sampling stratifié pour visualisations — évite le crash avec 250K points.
    """
    n = n or CONFIG['viz_sample_size']
    if len(df) <= n:
        return df
    
    if stratify_col:
        # Sampling stratifié par quintile pour représentativité
        df['_strat'] = pd.qcut(df[stratify_col], q=5, labels=False, duplicates='drop')
        sampled = df.groupby('_strat', group_keys=False).apply(
            lambda x: x.sample(min(len(x), n // 5), random_state=CONFIG['random_state'])
        ).drop('_strat', axis=1)
        df.drop('_strat', axis=1, inplace=True)
        return sampled.sample(min(len(sampled), n), random_state=CONFIG['random_state'])
    
    return df.sample(n, random_state=CONFIG['random_state'])


# ─── Stats descriptives ────────────────────────────────────────────────────────
print('=' * 60)
print('📊 STATISTIQUES DESCRIPTIVES')
print('=' * 60)
stats = df[CONFIG['target_cols']].describe().round(3)
print(stats)

# ─── Figure 1 : Distributions des propriétés ──────────────────────────────────
sample_viz = sample_for_viz(df, CONFIG['viz_sample_size'], 'logP')
print(f'\n🎨 Visualisation sur échantillon : {len(sample_viz):,} molécules')

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Distribution des Propriétés Physico-Chimiques\n'
             f'(Échantillon stratifié : {len(sample_viz):,}/{len(df):,} molécules)',
             fontsize=14, fontweight='bold', y=1.02)

props = CONFIG['target_cols']
for i, prop in enumerate(props):
    ax_hist = axes[0, i]
    ax_box = axes[1, i]
    color = list(COLORS.values())[i]
    
    # Histogramme + KDE
    data = sample_viz[prop].dropna()
    ax_hist.hist(data, bins=50, color=color, alpha=0.7, edgecolor='white', linewidth=0.5)
    ax_hist.set_title(f'{prop} — Distribution', fontweight='bold')
    ax_hist.set_xlabel(prop)
    ax_hist.set_ylabel('Fréquence')
    
    # Stats sur le plot
    mu, sigma = data.mean(), data.std()
    ax_hist.axvline(mu, color='red', linestyle='--', alpha=0.8, label=f'μ={mu:.2f}')
    ax_hist.legend(fontsize=9)
    
    # Boxplot
    bp = ax_box.boxplot(data, vert=True, patch_artist=True,
                        boxprops=dict(facecolor=color, alpha=0.6),
                        medianprops=dict(color='red', linewidth=2))
    ax_box.set_title(f'{prop} — Boxplot')
    ax_box.set_ylabel(prop)
    ax_box.text(1.1, mu, f'  σ={sigma:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f"{CONFIG['figures_dir']}01_distributions.png", dpi=120, bbox_inches='tight')
plt.show()
plt.close()  # ← TOUJOURS fermer après show()
del sample_viz; gc.collect()
print('✅ Figure 1 sauvegardée')

📊 STATISTIQUES DESCRIPTIVES
             logP         qed         SAS
count  234783.000  234783.000  234783.000
mean        2.483       0.735       3.029
std         1.319       0.130       0.770
min        -1.486       0.341       1.765
25%         1.612       0.655       2.424
50%         2.612       0.764       2.890
75%         3.467       0.836       3.513
max         5.158       0.930       5.380

🎨 Visualisation sur échantillon : 5,000 molécules
✅ Figure 1 sauvegardée


In [20]:
# ─── Figure 2 : Corrélations ───────────────────────────────────────────────────
sample_corr = sample_for_viz(df, 10_000)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Analyse des Corrélations entre Propriétés', fontsize=14, fontweight='bold')

# Heatmap
corr_matrix = sample_corr[CONFIG['target_cols']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, ax=axes[0], annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, square=True, linewidths=2,
            cbar_kws={'shrink': 0.8})
axes[0].set_title('Matrice de Corrélation')

# Scatter logP vs QED (coloré par SAS)
scatter = axes[1].scatter(
    sample_corr['logP'], sample_corr['qed'],
    c=sample_corr['SAS'], cmap='viridis',
    alpha=0.3, s=5, rasterized=True  # ← rasterized=True réduit la taille du fichier
)
plt.colorbar(scatter, ax=axes[1], label='SAS')
axes[1].set_xlabel('logP')
axes[1].set_ylabel('QED')
axes[1].set_title('logP vs QED (coloré par SAS)')

# Zone drug-like (Lipinski)
axes[1].axvline(x=5, color='red', linestyle='--', alpha=0.5, label='logP=5 (Lipinski)')
axes[1].axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='QED=0.6')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{CONFIG['figures_dir']}02_correlations.png", dpi=120, bbox_inches='tight')
plt.show()
plt.close()
del sample_corr; gc.collect()
print('✅ Figure 2 sauvegardée')

✅ Figure 2 sauvegardée


## 🔬 4. Calcul des Fingerprints Moléculaires (par Chunks)

In [23]:
import os

# Supprime l'ancien cache
for cache_file in [
    Path(CONFIG['output_dir']) / 'fingerprints.npy',
    Path(CONFIG['output_dir']) / 'descriptors.parquet'
]:
    if cache_file.exists():
        os.remove(cache_file)
        print(f'🗑️  Supprimé : {cache_file}')

print('✅ Cache nettoyé — relance la cellule des fingerprints')

🗑️  Supprimé : outputs\fingerprints.npy
🗑️  Supprimé : outputs\descriptors.parquet
✅ Cache nettoyé — relance la cellule des fingerprints


In [24]:
def smiles_to_fingerprint(smiles: str, radius: int = 2, nbits: int = 1024) -> np.ndarray:
    """
    Convertit un SMILES en fingerprint Morgan (vecteur binaire).
    Retourne un array de zéros si le SMILES est invalide.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(nbits, dtype=np.uint8)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)
    arr = np.zeros(nbits, dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def compute_fingerprints_chunked(
    smiles_series: pd.Series,
    chunk_size: int = 10_000,
    radius: int = 2,
    nbits: int = 1024,
    n_jobs: int = -1
) -> np.ndarray:
    """
    Calcul des fingerprints par chunks avec joblib.Parallel.
    Évite le crash mémoire sur 250K molécules.
    """
    from joblib import Parallel, delayed
    
    print(f'🔬 Calcul des fingerprints Morgan (r={radius}, {nbits} bits)')
    print(f'   Molécules : {len(smiles_series):,} | Chunks : {chunk_size:,}')
    
    chunks = []
    n_chunks = (len(smiles_series) + chunk_size - 1) // chunk_size
    
    for i in tqdm(range(0, len(smiles_series), chunk_size), total=n_chunks, desc='   Chunks'):
        chunk_smiles = smiles_series.iloc[i:i+chunk_size].tolist()
        
        # Parallélisation intra-chunk
        fps = Parallel(n_jobs=n_jobs, backend='threading')(
            delayed(smiles_to_fingerprint)(s, radius, nbits) for s in chunk_smiles
        )
        chunks.append(np.array(fps, dtype=np.uint8))
        
        # Libération mémoire
        del fps
    
    result = np.vstack(chunks)
    del chunks; gc.collect()
    print(f'✅ Fingerprints calculés : {result.shape} | RAM : {result.nbytes / 1e6:.1f} MB')
    return result


def compute_rdkit_descriptors_chunked(
    smiles_series: pd.Series,
    chunk_size: int = 5_000
) -> pd.DataFrame:
    """
    Calcule un sous-ensemble de descripteurs RDKit pertinents.
    """
    DESCRIPTORS = [
        ('MolWt', Descriptors.MolWt),
        ('NumHDonors', rdMolDescriptors.CalcNumHBD),
        ('NumHAcceptors', rdMolDescriptors.CalcNumHBA),
        ('NumRotatableBonds', rdMolDescriptors.CalcNumRotatableBonds),
        ('NumAromaticRings', rdMolDescriptors.CalcNumAromaticRings),
        ('TPSA', Descriptors.TPSA),
        ('NumRings', rdMolDescriptors.CalcNumRings),
        ('FractionCSP3', rdMolDescriptors.CalcFractionCSP3),
        ('NumStereocenters', rdMolDescriptors.CalcNumAtomStereoCenters),
        ('MolLogP', Descriptors.MolLogP),
    ]
    
    print(f'🔬 Calcul des descripteurs RDKit ({len(DESCRIPTORS)} features)...')
    all_rows = []
    desc_names = [d[0] for d in DESCRIPTORS]
    
    for i in tqdm(range(0, len(smiles_series), chunk_size), desc='   Descripteurs'):
        chunk = smiles_series.iloc[i:i+chunk_size]
        rows = []
        for smi in chunk:
            mol = Chem.MolFromSmiles(str(smi))
            if mol:
                rows.append([fn(mol) for _, fn in DESCRIPTORS])
            else:
                rows.append([np.nan] * len(DESCRIPTORS))
        all_rows.extend(rows)
    
    desc_df = pd.DataFrame(all_rows, columns=desc_names)
    desc_df = desc_df.fillna(desc_df.median())
    print(f'✅ Descripteurs calculés : {desc_df.shape}')
    return desc_df


# ─── Calcul sur le dataset complet ─────────────────────────────────────────────
# Fingerprints (sauvegardés pour réutilisation)
fp_cache = Path(CONFIG['output_dir']) / 'fingerprints.npy'
desc_cache = Path(CONFIG['output_dir']) / 'descriptors.parquet'

if fp_cache.exists() and desc_cache.exists():
    print('💾 Chargement depuis le cache...')
    X_fp = np.load(fp_cache)
    df_desc = pd.read_parquet(desc_cache)
    print(f'✅ Cache chargé : FP={X_fp.shape}, Desc={df_desc.shape}')
else:
    X_fp = compute_fingerprints_chunked(
        df[CONFIG['smiles_col']],
        chunk_size=CONFIG['fingerprint_chunk_size'],
        nbits=CONFIG['fp_nbits']
    )
    df_desc = compute_rdkit_descriptors_chunked(
        df[CONFIG['smiles_col']],
        chunk_size=5_000
    )
    # Sauvegarde cache
    np.save(fp_cache, X_fp)
    df_desc.to_parquet(desc_cache, index=False)
    print('💾 Cache sauvegardé')

# Feature matrix combinée
X_combined = np.hstack([X_fp, df_desc.values.astype(np.float32)])
print(f'\n📐 Feature matrix : {X_combined.shape}')

🔬 Calcul des fingerprints Morgan (r=2, 1024 bits)
   Molécules : 234,783 | Chunks : 10,000


   Chunks: 100%|██████████| 24/24 [04:35<00:00, 11.48s/it]


✅ Fingerprints calculés : (234783, 1024) | RAM : 240.4 MB
🔬 Calcul des descripteurs RDKit (10 features)...


   Descripteurs: 100%|██████████| 47/47 [07:54<00:00, 10.09s/it]


✅ Descripteurs calculés : (234783, 10)
💾 Cache sauvegardé

📐 Feature matrix : (234783, 1034)


## 🗺️ 5. Exploration de l'Espace Chimique (PCA + t-SNE)

In [25]:
import gc
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ─── 1. PCA ───────────────────────────────────────────────────────────────────
available = len(X_fp)
pca_sample_size  = min(CONFIG['pca_sample_size'],  available)
tsne_sample_size = min(CONFIG['tsne_sample_size'], pca_sample_size)

# ← CORRECTION PRINCIPALE : n_components ne peut pas dépasser min(n_samples, n_features)
max_components = min(pca_sample_size, X_fp.shape[1])
n_components_50 = min(50, max_components)
n_components_2  = min(2,  max_components)

print(f'🗺️  PCA sur {pca_sample_size:,} molécules (disponibles : {available:,})')
print(f'   X_fp shape : {X_fp.shape} → n_components utilisés : {n_components_50}')

pca_idx     = np.random.choice(available, pca_sample_size, replace=False)
X_pca_input = X_fp[pca_idx].astype(np.float32)

# PCA nD → base pour t-SNE
pca      = PCA(n_components=n_components_50, random_state=CONFIG['random_state'])
X_pca_50 = pca.fit_transform(X_pca_input).astype(np.float32)

# PCA 2D → visualisation directe
pca_2d   = PCA(n_components=n_components_2, random_state=CONFIG['random_state'])
X_pca_2d = pca_2d.fit_transform(X_pca_input).astype(np.float32)
print(f'   Variance expliquée (PC1+PC2) : {pca_2d.explained_variance_ratio_.sum()*100:.1f}%')

del X_pca_input
gc.collect()

# ─── 2. t-SNE ─────────────────────────────────────────────────────────────────
# t-SNE nécessite n_components_50 > 2 pour être utile ; fallback sur PCA 2D sinon
use_tsne = n_components_50 > 2

if use_tsne:
    print(f'🗺️  t-SNE sur {tsne_sample_size:,} molécules...')
    tsne_idx     = pca_idx[:tsne_sample_size]
    X_tsne_input = X_pca_50[:tsne_sample_size]

    t0 = time.time()
    tsne = TSNE(
    n_components=2,
    perplexity=min(30, tsne_sample_size - 1),
    max_iter=500,          # ← n_iter renommé en max_iter (sklearn >= 1.4)
    random_state=CONFIG['random_state'],
    n_jobs=1,
    method='barnes_hut',
    verbose=0
    )   
    X_tsne = tsne.fit_transform(X_tsne_input).astype(np.float32)
    print(f'   t-SNE terminé en {time.time()-t0:.1f}s')
    del X_tsne_input
else:
    print('⚠️  Trop peu de features pour t-SNE → PCA 2D utilisée pour tous les graphes')
    tsne_idx = pca_idx[:tsne_sample_size]
    X_tsne   = X_pca_2d[:tsne_sample_size]   # réutilise PCA 2D

del X_pca_50
gc.collect()

# ─── 3. Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Exploration de l'Espace Chimique", fontsize=14, fontweight='bold')

color_props = ['logP', 'qed', 'SAS']

for ax, prop in zip(axes, color_props):
    prop_vals_pca  = df[prop].values[pca_idx]
    prop_vals_tsne = df[prop].values[tsne_idx]

    if prop != 'SAS':
        sc = ax.scatter(
            X_tsne[:, 0], X_tsne[:, 1],
            c=prop_vals_tsne, cmap='viridis',
            s=3, alpha=0.5, rasterized=True
        )
        label = 't-SNE' if use_tsne else 'PCA'
        ax.set_title(f'{label} coloré par {prop}', fontweight='bold')
    else:
        sc = ax.scatter(
            X_pca_2d[:, 0], X_pca_2d[:, 1],
            c=prop_vals_pca, cmap='plasma',
            s=3, alpha=0.5, rasterized=True
        )
        ax.set_title(f'PCA coloré par {prop}', fontweight='bold')

    plt.colorbar(sc, ax=ax, shrink=0.7, label=prop)
    ax.set_xlabel('Dim 1')
    ax.set_ylabel('Dim 2')

plt.tight_layout()
plt.savefig(f"{CONFIG['figures_dir']}03_chemical_space.png", dpi=120, bbox_inches='tight')
plt.show()
plt.close()

# ─── 4. Nettoyage RAM ─────────────────────────────────────────────────────────
del X_tsne, X_pca_2d
gc.collect()
print('✅ Figure 3 sauvegardée et RAM entièrement libérée')

🗺️  PCA sur 10,000 molécules (disponibles : 234,783)
   X_fp shape : (234783, 1024) → n_components utilisés : 50
   Variance expliquée (PC1+PC2) : 4.7%
🗺️  t-SNE sur 3,000 molécules...
   t-SNE terminé en 7.9s
✅ Figure 3 sauvegardée et RAM entièrement libérée


## 🤖 6. Modèles ML — Prédiction des Propriétés

In [26]:
def train_and_evaluate(
    X: np.ndarray,
    y: np.ndarray,
    target_name: str,
    models: dict = None
) -> dict:
    """
    Entraîne et évalue plusieurs modèles ML pour une propriété cible.
    Retourne un dictionnaire de résultats.
    """
    print(f'\n🤖 Entraînement pour : {target_name}')
    print(f'   Données : {X.shape[0]:,} × {X.shape[1]} features')
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=CONFIG['test_size'],
        random_state=CONFIG['random_state']
    )
    print(f'   Train: {len(X_train):,} | Test: {len(X_test):,}')
    
    if models is None:
        models = {
            'XGBoost': xgb.XGBRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=CONFIG['random_state'],
                tree_method='hist',  # ← Rapide sur grandes données
                n_jobs=-1,
                verbosity=0
            ),
            'LightGBM': lgb.LGBMRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=CONFIG['random_state'],
                n_jobs=-1,
                verbose=-1
            ),
            'RandomForest': RandomForestRegressor(
                n_estimators=200,
                max_depth=15,
                min_samples_split=5,
                random_state=CONFIG['random_state'],
                n_jobs=-1,
                max_features='sqrt'  # ← Réduit l'overfitting
            )
        }
    
    results = {}
    best_model = None
    best_r2 = -np.inf
    
    for name, model in models.items():
        t0 = time.time()
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        elapsed = time.time() - t0
        
        results[name] = {
            'model': model,
            'r2': r2,
            'rmse': rmse,
            'mae': mae,
            'y_test': y_test,
            'y_pred': y_pred,
            'time': elapsed
        }
        
        print(f'   {name:15s} | R²={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f} | {elapsed:.1f}s')
        
        if r2 > best_r2:
            best_r2 = r2
            best_model = name
    
    print(f'   🏆 Meilleur modèle : {best_model} (R²={best_r2:.4f})')
    
    # Sauvegarder le meilleur modèle
    joblib.dump(
        results[best_model]['model'],
        f"{CONFIG['models_dir']}{target_name}_best_model.pkl"
    )
    
    results['best'] = best_model
    results['X_test'] = X_test
    results['y_test_arr'] = y_test
    return results


# ─── Scaler ────────────────────────────────────────────────────────────────────
scaler = RobustScaler()  # Robuste aux outliers
# Utiliser uniquement les descripteurs pour le scaling (les FP sont binaires)
X_desc_scaled = scaler.fit_transform(df_desc.values.astype(np.float32))

# Combiner FP binaires + descripteurs scalés
X_final = np.hstack([X_fp, X_desc_scaled]).astype(np.float32)
print(f'📐 Feature matrix finale : {X_final.shape}')

# ─── Entraîner un modèle par propriété ────────────────────────────────────────
all_results = {}
for target in CONFIG['target_cols']:
    y = df[target].values.astype(np.float32)
    all_results[target] = train_and_evaluate(X_final, y, target)

gc.collect()
print('\n✅ Tous les modèles entraînés et sauvegardés')

📐 Feature matrix finale : (234783, 1034)

🤖 Entraînement pour : logP
   Données : 234,783 × 1034 features
   Train: 187,826 | Test: 46,957
   XGBoost         | R²=0.9998 | RMSE=0.0199 | MAE=0.0134 | 116.5s
   LightGBM        | R²=0.9996 | RMSE=0.0267 | MAE=0.0190 | 15.6s
   RandomForest    | R²=0.7461 | RMSE=0.6677 | MAE=0.5024 | 106.5s
   🏆 Meilleur modèle : XGBoost (R²=0.9998)

🤖 Entraînement pour : qed
   Données : 234,783 × 1034 features
   Train: 187,826 | Test: 46,957
   XGBoost         | R²=0.9281 | RMSE=0.0347 | MAE=0.0206 | 73.0s
   LightGBM        | R²=0.9221 | RMSE=0.0362 | MAE=0.0217 | 11.7s
   RandomForest    | R²=0.6305 | RMSE=0.0787 | MAE=0.0606 | 128.1s
   🏆 Meilleur modèle : XGBoost (R²=0.9281)

🤖 Entraînement pour : SAS
   Données : 234,783 × 1034 features
   Train: 187,826 | Test: 46,957
   XGBoost         | R²=0.9521 | RMSE=0.1684 | MAE=0.1230 | 78.5s
   LightGBM        | R²=0.9481 | RMSE=0.1752 | MAE=0.1283 | 13.5s
   RandomForest    | R²=0.7722 | RMSE=0.3672 | MAE

In [27]:
# ─── Figure 4 : Scatter plots Prédictions vs Réalité ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Prédictions vs Valeurs Réelles — Meilleurs Modèles', fontsize=14, fontweight='bold')

for ax, target in zip(axes, CONFIG['target_cols']):
    res = all_results[target]
    best_name = res['best']
    y_test_plot = res[best_name]['y_test']
    y_pred_plot = res[best_name]['y_pred']
    r2 = res[best_name]['r2']
    rmse = res[best_name]['rmse']
    color = list(COLORS.values())[CONFIG['target_cols'].index(target)]
    
    # Sample pour le plot (évite 50K points)
    n_plot = min(3000, len(y_test_plot))
    idx_plot = np.random.choice(len(y_test_plot), n_plot, replace=False)
    
    ax.scatter(y_test_plot[idx_plot], y_pred_plot[idx_plot],
               alpha=0.3, s=10, color=color, rasterized=True)
    
    # Ligne parfaite
    lims = [min(y_test_plot.min(), y_pred_plot.min()),
            max(y_test_plot.max(), y_pred_plot.max())]
    ax.plot(lims, lims, 'r--', lw=2, alpha=0.8, label='Parfait')
    
    ax.set_xlabel(f'{target} Réel')
    ax.set_ylabel(f'{target} Prédit')
    ax.set_title(f'{target} ({best_name})\nR²={r2:.4f} | RMSE={rmse:.4f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{CONFIG['figures_dir']}04_predictions.png", dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print('✅ Figure 4 sauvegardée')

✅ Figure 4 sauvegardée


In [30]:
from scipy.stats import pearsonr
r, _ = pearsonr(df['logP'].values, df_desc['MolLogP'].values)
print(f"Corrélation logP ↔ MolLogP : {r:.4f}")

Corrélation logP ↔ MolLogP : 1.0000


In [31]:
df_desc_clean = df_desc.drop(columns=['MolLogP'])
X_final = np.hstack([X_fp, df_desc_clean.values.astype(np.float32)])
print(f"Shape : {X_final.shape}")  # → (234783, 1033)

Shape : (234783, 1033)


In [32]:
all_results = {}
for target in CONFIG['target_cols']:
    y = df[target].values.astype(np.float32)
    all_results[target] = train_and_evaluate(X_final, y, target)

gc.collect()
print('\n✅ Tous les modèles entraînés et sauvegardés')


🤖 Entraînement pour : logP
   Données : 234,783 × 1033 features
   Train: 187,826 | Test: 46,957
   XGBoost         | R²=0.9354 | RMSE=0.3369 | MAE=0.2540 | 84.5s
   LightGBM        | R²=0.9309 | RMSE=0.3484 | MAE=0.2625 | 14.1s
   RandomForest    | R²=0.5617 | RMSE=0.8773 | MAE=0.7032 | 98.4s
   🏆 Meilleur modèle : XGBoost (R²=0.9354)

🤖 Entraînement pour : qed
   Données : 234,783 × 1033 features
   Train: 187,826 | Test: 46,957
   XGBoost         | R²=0.9121 | RMSE=0.0384 | MAE=0.0255 | 60.2s
   LightGBM        | R²=0.9071 | RMSE=0.0395 | MAE=0.0263 | 7.7s
   RandomForest    | R²=0.6124 | RMSE=0.0806 | MAE=0.0625 | 87.5s
   🏆 Meilleur modèle : XGBoost (R²=0.9121)

🤖 Entraînement pour : SAS
   Données : 234,783 × 1033 features
   Train: 187,826 | Test: 46,957
   XGBoost         | R²=0.9475 | RMSE=0.1763 | MAE=0.1275 | 58.3s
   LightGBM        | R²=0.9433 | RMSE=0.1832 | MAE=0.1326 | 7.3s
   RandomForest    | R²=0.7524 | RMSE=0.3829 | MAE=0.2986 | 80.6s
   🏆 Meilleur modèle : XGBoost

In [33]:
for target, res in all_results.items():
    if not res:
        continue
    best = res['best']
    model = res[best]['model']
    
    r2_test = res[best]['r2']
    y_train_full = df[target].values.astype(np.float32)
    r2_train = r2_score(y_train_full, model.predict(X_final))
    
    gap = r2_train - r2_test
    status = "⚠️ Overfitting" if gap > 0.05 else "✅ OK"
    print(f"{target:6s} → Train: {r2_train:.4f} | Test: {r2_test:.4f} | Gap: {gap:.4f} {status}")

logP   → Train: 0.9426 | Test: 0.9354 | Gap: 0.0073 ✅ OK
qed    → Train: 0.9247 | Test: 0.9121 | Gap: 0.0126 ✅ OK
SAS    → Train: 0.9538 | Test: 0.9475 | Gap: 0.0063 ✅ OK


In [45]:
# Test sur une seule propriété avec affichage complet
target = 'logP'
y = df[target].values.astype(np.float32)
print(f"y shape : {y.shape}")
print(f"y dtype : {y.dtype}")
print(f"y exemple : {y[:5]}")

result = train_and_evaluate(X_final, y, target)
print(f"Résultat : {result}")

y shape : (234783,)
y dtype : float32
y exemple : [5.0506  3.1137  4.96778 4.00022 3.60956]

🤖 Entraînement pour : logP
   Données : 234,783 × 1033 features
   Train: 187,826 | Test: 46,957
   XGBoost         | R²=0.9354 | RMSE=0.3369 | MAE=0.2540 | 67.3s
   LightGBM        | R²=0.9309 | RMSE=0.3484 | MAE=0.2625 | 8.5s
   RandomForest    | R²=0.5617 | RMSE=0.8773 | MAE=0.7032 | 83.9s
   🏆 Meilleur modèle : XGBoost (R²=0.9354)
Résultat : {'XGBoost': {'model': XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_de

In [46]:
all_results = {}
for target in CONFIG['target_cols']:
    y = df[target].values.astype(np.float32)
    all_results[target] = train_and_evaluate(X_final, y, target)

gc.collect()
print(f'\n✅ Terminé : {list(all_results.keys())}')


🤖 Entraînement pour : logP
   Données : 234,783 × 1033 features
   Train: 187,826 | Test: 46,957
   XGBoost         | R²=0.9354 | RMSE=0.3369 | MAE=0.2540 | 66.8s
   LightGBM        | R²=0.9309 | RMSE=0.3484 | MAE=0.2625 | 8.7s
   RandomForest    | R²=0.5617 | RMSE=0.8773 | MAE=0.7032 | 81.2s
   🏆 Meilleur modèle : XGBoost (R²=0.9354)

🤖 Entraînement pour : qed
   Données : 234,783 × 1033 features
   Train: 187,826 | Test: 46,957
   XGBoost         | R²=0.9121 | RMSE=0.0384 | MAE=0.0255 | 57.6s
   LightGBM        | R²=0.9071 | RMSE=0.0395 | MAE=0.0263 | 7.0s
   RandomForest    | R²=0.6124 | RMSE=0.0806 | MAE=0.0625 | 82.6s
   🏆 Meilleur modèle : XGBoost (R²=0.9121)

🤖 Entraînement pour : SAS
   Données : 234,783 × 1033 features
   Train: 187,826 | Test: 46,957
   XGBoost         | R²=0.9475 | RMSE=0.1763 | MAE=0.1275 | 61.6s
   LightGBM        | R²=0.9433 | RMSE=0.1832 | MAE=0.1326 | 10.9s
   RandomForest    | R²=0.7524 | RMSE=0.3829 | MAE=0.2986 | 87.4s
   🏆 Meilleur modèle : XGBoost

In [47]:
# Relance SHAP
for target in CONFIG['target_cols']:
    best_name = all_results[target]['best']
    best_model = all_results[target][best_name]['model']
    X_test_target = all_results[target]['X_test'].astype(np.float32)
    
    try:
        compute_shap_analysis(
            best_model, X_test_target, feature_names,
            target, n_samples=300
        )
    except Exception as e:
        print(f'⚠️  SHAP {target} échoué : {e}')

💡 SHAP pour logP (n=300)...
⚠️  SHAP logP échoué : could not convert string to float: '[2.4844913E0]'
💡 SHAP pour qed (n=300)...
⚠️  SHAP qed échoué : could not convert string to float: '[7.3467094E-1]'
💡 SHAP pour SAS (n=300)...
⚠️  SHAP SAS échoué : could not convert string to float: '[3.0279775E0]'


In [34]:
# ─── Figure 5 : Comparaison des modèles ───────────────────────────────────────
model_names = ['XGBoost', 'LightGBM', 'RandomForest']
metrics = ['r2', 'rmse', 'mae']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Comparaison des Modèles ML', fontsize=14, fontweight='bold')

for ax, metric in zip(axes, metrics):
    x = np.arange(len(CONFIG['target_cols']))
    width = 0.25
    colors_bar = ['#3B82F6', '#10B981', '#F59E0B']
    
    for j, model_name in enumerate(model_names):
        values = [all_results[t][model_name][metric] for t in CONFIG['target_cols']]
        bars = ax.bar(x + j*width, values, width, label=model_name,
                      color=colors_bar[j], alpha=0.8, edgecolor='white')
        
        # Valeurs sur les barres
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=7)
    
    ax.set_xticks(x + width)
    ax.set_xticklabels(CONFIG['target_cols'])
    ax.set_title(metric.upper())
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CONFIG['figures_dir']}05_model_comparison.png", dpi=120, bbox_inches='tight')
plt.show()
plt.close()
print('✅ Figure 5 sauvegardée')

✅ Figure 5 sauvegardée


## 💡 7. Interprétabilité avec SHAP

In [48]:
def compute_shap_analysis(
    model,
    X_test: np.ndarray,
    feature_names: list,
    target_name: str,
    n_samples: int = 500  # ← SHAP est lent → petit échantillon
):
    """
    Analyse SHAP pour comprendre l'importance des features.
    Limité à n_samples pour éviter le crash.
    """
    print(f'💡 SHAP pour {target_name} (n={n_samples})...')
    
    idx = np.random.choice(len(X_test), min(n_samples, len(X_test)), replace=False)
    X_shap = X_test[idx]
    
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_shap)
    
    # Importance des features (top 20 descripteurs seulement)
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    # Top features parmi les descripteurs RDKit (dernières colonnes)
    n_fp = CONFIG['fp_nbits']
    desc_importances = mean_abs_shap[n_fp:]
    fp_total = mean_abs_shap[:n_fp].sum()
    
    desc_names = list(df_desc.columns)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'Analyse SHAP — {target_name}', fontsize=13, fontweight='bold')
    
    # Barplot importance descripteurs
    sorted_idx = np.argsort(desc_importances)[::-1]
    axes[0].barh(
        [desc_names[i] for i in sorted_idx],
        desc_importances[sorted_idx],
        color=COLORS.get(target_name, COLORS['accent']),
        alpha=0.8
    )
    axes[0].set_title('Importance des Descripteurs RDKit')
    axes[0].set_xlabel('|SHAP| moyen')
    axes[0].invert_yaxis()
    
    # Comparaison FP vs Descripteurs
    axes[1].bar(
        ['Morgan FP\n(1024 bits total)', 'Descripteurs RDKit\n(10 features)'],
        [fp_total, desc_importances.sum()],
        color=[COLORS['logP'], COLORS['qed']],
        alpha=0.8, edgecolor='white'
    )
    axes[1].set_title('Contribution : FP vs Descripteurs')
    axes[1].set_ylabel('|SHAP| total')
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['figures_dir']}shap_{target_name}.png", dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    
    del shap_values; gc.collect()
    print(f'✅ SHAP {target_name} terminé')


# ─── SHAP pour chaque propriété ────────────────────────────────────────────────
feature_names = ([f'FP_{i}' for i in range(CONFIG['fp_nbits'])] +
                 list(df_desc.columns))

for target in CONFIG['target_cols']:
    best_name = all_results[target]['best']
    best_model = all_results[target][best_name]['model']
    X_test_target = all_results[target]['X_test']
    
    try:
        compute_shap_analysis(
            best_model, X_test_target, feature_names,
            target, n_samples=300
        )
    except Exception as e:
        print(f'⚠️  SHAP {target} échoué : {e}')

gc.collect()

💡 SHAP pour logP (n=300)...
⚠️  SHAP logP échoué : could not convert string to float: '[2.4844913E0]'
💡 SHAP pour qed (n=300)...
⚠️  SHAP qed échoué : could not convert string to float: '[7.3467094E-1]'
💡 SHAP pour SAS (n=300)...
⚠️  SHAP SAS échoué : could not convert string to float: '[3.0279775E0]'


0

## 🧬 8. Génération de Molécules (MolMIM / Perturbation Latente)

In [ ]:
def drug_likeness_filter(smiles: str) -> dict:
    """
    Filtre de Lipinski (Rule of Five) + critères supplémentaires.
    Retourne un dictionnaire avec les propriétés et le verdict.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {'valid': False}
    
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    tpsa = Descriptors.TPSA(mol)
    rot_bonds = rdMolDescriptors.CalcNumRotatableBonds(mol)
    
    # Règles de Lipinski
    lipinski_ok = (mw <= 500) and (logp <= 5) and (hbd <= 5) and (hba <= 10)
    
    # Règle de Veber (biodisponibilité orale)
    veber_ok = (tpsa <= 140) and (rot_bonds <= 10)
    
    return {
        'valid': True,
        'MW': round(mw, 2),
        'logP': round(logp, 3),
        'HBD': hbd,
        'HBA': hba,
        'TPSA': round(tpsa, 2),
        'RotBonds': rot_bonds,
        'Lipinski': lipinski_ok,
        'Veber': veber_ok,
        'drug_like': lipinski_ok and veber_ok
    }


def generate_analogs_by_mutation(
    smiles: str,
    n_attempts: int = 100,
    max_valid: int = 20
) -> list:
    """
    Génère des analogues structuraux par mutations SMILES légères.
    Stratégie : substitutions d'atomes, ajout/suppression de groupes fonctionnels.
    """
    import re, random
    
    SUBSTITUTIONS = {
        'c': ['c', 'n'],      # Aromatique C → N (pyridine)
        'CC': ['CC', 'CF', 'CCl', 'CBr'],  # Alkyl
        'N': ['N', 'O', 'S'],  # Hétéroatomes
        'F': ['F', 'Cl', 'Br'],  # Halogènes
        'C(=O)': ['C(=O)', 'S(=O)', 'C(=S)'],  # Carbonyle
    }
    
    analogs = []
    for _ in range(n_attempts):
        modified = smiles
        for original, replacements in SUBSTITUTIONS.items():
            if original in modified and random.random() < 0.3:
                replacement = random.choice(replacements)
                modified = modified.replace(original, replacement, 1)
        
        if modified != smiles:
            mol = Chem.MolFromSmiles(modified)
            if mol is not None:
                canonical = Chem.MolToSmiles(mol)
                if canonical not in [a['smiles'] for a in analogs]:
                    analogs.append({'smiles': canonical, 'parent': smiles})
                    if len(analogs) >= max_valid:
                        break
    
    return analogs


def score_and_rank_candidates(
    candidates: list,
    models_dict: dict,  # {target: model}
    X_fp_func,          # Fonction de calcul fingerprint
    X_desc_func,        # Fonction de calcul descripteurs
    scaler: object
) -> pd.DataFrame:
    """
    Score les candidats avec les modèles ML et les règles drug-likeness.
    """
    smiles_list = [c['smiles'] for c in candidates]
    
    # Calcul des features
    fps = np.array([smiles_to_fingerprint(s, CONFIG['fp_radius'], CONFIG['fp_nbits'])
                    for s in smiles_list])
    descs_df = compute_rdkit_descriptors_chunked(pd.Series(smiles_list), chunk_size=500)
    descs_scaled = scaler.transform(descs_df.values.astype(np.float32))
    X = np.hstack([fps, descs_scaled]).astype(np.float32)
    
    # Prédictions
    results = pd.DataFrame({'smiles': smiles_list})
    for target, model in models_dict.items():
        results[f'{target}_pred'] = model.predict(X).round(4)
    
    # Drug-likeness
    dl_results = [drug_likeness_filter(s) for s in smiles_list]
    results['MW'] = [r.get('MW', np.nan) for r in dl_results]
    results['drug_like'] = [r.get('drug_like', False) for r in dl_results]
    results['Lipinski'] = [r.get('Lipinski', False) for r in dl_results]
    
    # Score composite (QED élevé, SAS faible, logP dans [1-5])
    if 'qed_pred' in results.columns and 'SAS_pred' in results.columns:
        results['composite_score'] = (
            results['qed_pred'] * 0.4 +
            (1 / (1 + results['SAS_pred'])) * 0.3 +
            results['logP_pred'].clip(1, 5).apply(lambda x: 1 - abs(x - 3) / 2) * 0.3
        ).round(4)
    
    return results.sort_values('composite_score', ascending=False)


print('✅ Fonctions de génération définies')

In [ ]:
# ─── Génération : sélectionner les meilleures molécules seeds ─────────────────
print('🧬 Sélection des molécules seeds (meilleures QED + Lipinski)...')

# Filtrer les drug-like dans le dataset
seed_sample = df.sample(min(5000, len(df)), random_state=42)
dl_results = [drug_likeness_filter(str(s)) for s in tqdm(seed_sample[CONFIG['smiles_col']], desc='Filtre Lipinski')]

seed_sample = seed_sample.copy()
seed_sample['drug_like'] = [r.get('drug_like', False) for r in dl_results]
drug_like_seeds = seed_sample[seed_sample['drug_like']].nlargest(20, 'qed')

print(f'✅ {len(drug_like_seeds)} molécules seeds sélectionnées (drug-like + QED élevé)')

# ─── Génération des analogues ──────────────────────────────────────────────────
print('\n🧬 Génération des analogues...')
all_candidates = []

for _, row in tqdm(drug_like_seeds.head(10).iterrows(), total=10, desc='Seeds'):
    smiles = str(row[CONFIG['smiles_col']])
    analogs = generate_analogs_by_mutation(smiles, n_attempts=50, max_valid=10)
    all_candidates.extend(analogs)

print(f'✅ {len(all_candidates)} candidats générés')

# ─── Scoring des candidats ────────────────────────────────────────────────────
if len(all_candidates) > 0:
    models_for_scoring = {}
    for target in CONFIG['target_cols']:
        best_name = all_results[target]['best']
        models_for_scoring[target] = all_results[target][best_name]['model']
    
    print('\n📊 Scoring des candidats...')
    df_candidates = score_and_rank_candidates(
        all_candidates,
        models_for_scoring,
        X_fp_func=smiles_to_fingerprint,
        X_desc_func=compute_rdkit_descriptors_chunked,
        scaler=scaler
    )
    
    # Filtrer drug-like uniquement
    df_candidates_filtered = df_candidates[df_candidates['drug_like']].copy()
    
    print(f'\n🏆 TOP 10 MOLÉCULES CANDIDATES (drug-like)')
    print('=' * 80)
    cols_show = ['smiles', 'logP_pred', 'qed_pred', 'SAS_pred', 'MW', 'composite_score']
    cols_show = [c for c in cols_show if c in df_candidates_filtered.columns]
    print(df_candidates_filtered[cols_show].head(10).to_string(index=False))
    
    # Sauvegarder
    output_path = f"{CONFIG['output_dir']}top_candidates.csv"
    df_candidates_filtered.head(50).to_csv(output_path, index=False)
    print(f'\n💾 Top candidats sauvegardés : {output_path}')
else:
    print('⚠️  Aucun candidat valide généré')

## 📈 9. Visualisation des Candidats Générés

In [ ]:
if len(all_candidates) > 0 and len(df_candidates_filtered) > 0:
    # ─── Figure 6 : Espace chimique — Dataset vs Candidats ────────────────────
    
    # FP des candidats
    cand_smiles = df_candidates_filtered['smiles'].tolist()[:30]
    cand_fps = np.array([smiles_to_fingerprint(s) for s in cand_smiles])
    
    # FP du dataset (échantillon)
    n_ref = 500
    ref_idx = np.random.choice(len(X_fp), n_ref, replace=False)
    ref_fps = X_fp[ref_idx]
    
    # PCA combinée
    all_fps = np.vstack([ref_fps, cand_fps])
    pca_viz = PCA(n_components=2, random_state=42)
    all_pca = pca_viz.fit_transform(all_fps)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Espace Chimique : Dataset vs Molécules Générées', fontsize=13, fontweight='bold')
    
    # Scatter espace chimique
    axes[0].scatter(all_pca[:n_ref, 0], all_pca[:n_ref, 1],
                    c='lightblue', s=20, alpha=0.5, label=f'Dataset ({n_ref})', rasterized=True)
    axes[0].scatter(all_pca[n_ref:, 0], all_pca[n_ref:, 1],
                    c='red', s=80, alpha=0.8, marker='*',
                    label=f'Candidats ({len(cand_fps)})', zorder=5)
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    axes[0].set_title('PCA — Espace Chimique')
    axes[0].legend()
    
    # Distribution composite score
    if 'composite_score' in df_candidates_filtered.columns:
        scores = df_candidates_filtered['composite_score'].dropna()
        axes[1].hist(scores, bins=20, color=COLORS['accent'],
                     alpha=0.8, edgecolor='white')
        axes[1].axvline(scores.mean(), color='red', linestyle='--',
                        label=f'Moyenne={scores.mean():.3f}')
        axes[1].set_xlabel('Score Composite')
        axes[1].set_ylabel('Nombre de candidats')
        axes[1].set_title('Distribution des Scores Composites')
        axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(f"{CONFIG['figures_dir']}06_candidates_analysis.png", dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    print('✅ Figure 6 sauvegardée')
    
    del cand_fps, ref_fps, all_fps; gc.collect()

## 🏁 10. Rapport Final

In [ ]:
print('=' * 65)
print('🏁 RAPPORT FINAL — Génération & Optimisation de Molécules par IA')
print('=' * 65)

print(f'\n📊 DATASET')
print(f'   Molécules totales analysées : {len(df):>10,}')
print(f'   Features (FP + Descripteurs): {X_final.shape[1]:>10,}')

print(f'\n🤖 PERFORMANCES DES MODÈLES')
print(f'   {"Propriété":<10} {"Meilleur modèle":<15} {"R²":>8} {"RMSE":>8} {"MAE":>8}')
print(f'   {"-"*55}')
for target in CONFIG['target_cols']:
    best_name = all_results[target]['best']
    r2 = all_results[target][best_name]['r2']
    rmse = all_results[target][best_name]['rmse']
    mae = all_results[target][best_name]['mae']
    print(f'   {target:<10} {best_name:<15} {r2:>8.4f} {rmse:>8.4f} {mae:>8.4f}')

if len(all_candidates) > 0:
    print(f'\n🧬 GÉNÉRATION')
    print(f'   Candidats générés        : {len(all_candidates):>10,}')
    print(f'   Drug-like (Lipinski+Veber): {len(df_candidates_filtered):>10,}')
    if 'composite_score' in df_candidates_filtered.columns:
        best_score = df_candidates_filtered['composite_score'].max()
        print(f'   Meilleur score composite : {best_score:>10.4f}')

print(f'\n📁 FICHIERS GÉNÉRÉS')
for d in [CONFIG['figures_dir'], CONFIG['models_dir'], CONFIG['output_dir']]:
    files = list(Path(d).glob('*'))
    print(f'   {d} : {len(files)} fichiers')
    for f in files:
        print(f'      - {f.name} ({f.stat().st_size/1024:.1f} KB)')

print('\n✅ Projet terminé avec succès!')
print('=' * 65)

---
## 🔧 Annexe : Intégration NVIDIA MolMIM (API)

Pour utiliser le vrai modèle **NVIDIA MolMIM** via NGC/BioNeMo :

In [ ]:
# ─── Intégration NVIDIA MolMIM (nécessite une clé API NGC) ────────────────────
# Documentation : https://docs.nvidia.com/bionemo/molmim/

MOLMIM_TEMPLATE = '''
import requests

NGC_API_KEY = "your_ngc_api_key_here"
MOLMIM_URL = "https://api.nvcf.nvidia.com/v2/nvcf/pexec/functions/"
FUNCTION_ID = "16c5f5e1-d8c6-4789-9abb-d4d03d0fd76e"  # MolMIM function ID

def molmim_generate(
    smiles_list: list,
    num_molecules: int = 10,
    iterations: int = 10,
    scaled_radius: float = 1.0
) -> list:
    """
    Génère de nouvelles molécules via NVIDIA MolMIM.
    
    Args:
        smiles_list    : Liste de SMILES seeds
        num_molecules  : Nombre de molécules à générer par seed
        iterations     : Iterations de recherche
        scaled_radius  : Rayon de perturbation dans l\'espace latent (0.1-3.0)
    """
    headers = {
        "Authorization": f"Bearer {NGC_API_KEY}",
        "Content-Type": "application/json",
        "Accept": "application/json"
    }
    
    all_generated = []
    
    for smi in smiles_list:
        payload = {
            "smi": smi,
            "num_molecules": num_molecules,
            "iterations": iterations,
            "scaled_radius": scaled_radius,
            "scaffold": "",  # optionnel : contrainte de scaffold
        }
        
        response = requests.post(
            MOLMIM_URL + FUNCTION_ID,
            headers=headers,
            json=payload,
            timeout=60
        )
        response.raise_for_status()
        result = response.json()
        all_generated.extend(result.get("generated", []))
    
    return all_generated


def molmim_encode(smiles: str) -> list:
    """Encode un SMILES en vecteur latent MolMIM."""
    headers = {"Authorization": f"Bearer {NGC_API_KEY}", "Content-Type": "application/json"}
    payload = {"smi": smiles, "action": "encode"}
    response = requests.post(MOLMIM_URL + FUNCTION_ID, headers=headers, json=payload)
    return response.json().get("encoding", [])
'''

print('📝 Template MolMIM API prêt.')
print('   → Remplacer NGC_API_KEY par votre vraie clé NGC')
print('   → Accès : https://catalog.ngc.nvidia.com/orgs/nvidia/teams/clara/models/molmim')

# Sauvegarder le template
with open(f"{CONFIG['output_dir']}molmim_api_template.py", 'w') as f:
    f.write(MOLMIM_TEMPLATE)
print('   → Template sauvegardé : outputs/molmim_api_template.py')